# 13 – Stats

Esplorazione e data cleaning del dataset `stats.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `mal_id` | ID dell'anime su MAL |
| `watching` | Numero di utenti che stanno guardando l'anime |
| `completed` | Numero di utenti che hanno completato l'anime |
| `on_hold` | Numero di utenti che hanno messo l'anime in pausa |
| `dropped` | Numero di utenti che hanno abbandonato l'anime |
| `plan_to_watch` | Numero di utenti che intendono guardare l'anime |
| `total` | Totale utenti che hanno l'anime in lista |
| `score_N_votes` | Numero di voti ricevuti per il punteggio N (1–10) |
| `score_N_percentage` | Percentuale di voti per il punteggio N rispetto al totale dei voti |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [229]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_stats = pd.read_csv('../datasets/stats.csv')
print(f'Shape: {df_stats.shape}')
df_stats.info()
df_stats.head()

Shape: (28955, 27)
<class 'pandas.DataFrame'>
RangeIndex: 28955 entries, 0 to 28954
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   mal_id               28955 non-null  int64  
 1   watching             28955 non-null  int64  
 2   completed            28955 non-null  int64  
 3   on_hold              28955 non-null  int64  
 4   dropped              28955 non-null  int64  
 5   plan_to_watch        28955 non-null  int64  
 6   total                28955 non-null  int64  
 7   score_1_votes        28525 non-null  float64
 8   score_1_percentage   28525 non-null  float64
 9   score_2_votes        28525 non-null  float64
 10  score_2_percentage   28525 non-null  float64
 11  score_3_votes        28525 non-null  float64
 12  score_3_percentage   28525 non-null  float64
 13  score_4_votes        28525 non-null  float64
 14  score_4_percentage   28525 non-null  float64
 15  score_5_votes        28525 n

,mal_id,watching,completed,on_hold,dropped,plan_to_watch,total,score_1_votes,score_1_percentage,score_2_votes,...,score_6_votes,score_6_percentage,score_7_votes,score_7_percentage,score_8_votes,score_8_percentage,score_9_votes,score_9_percentage,score_10_votes,score_10_percentage
0,59356,7,146,4,20,20,197,2.0,2.2,0.0,...,33.0,36.3,19.0,20.9,2.0,2.2,0.0,0.0,1.0,1.1
1,56036,21,770,8,29,113,941,5.0,1.0,6.0,...,138.0,27.4,144.0,28.6,81.0,16.1,17.0,3.4,40.0,8.0
2,2928,451,14953,302,349,6472,22527,101.0,1.0,93.0,...,2054.0,21.1,2709.0,27.8,1500.0,15.4,875.0,9.0,608.0,6.2
3,3269,726,22790,452,537,9762,34267,120.0,0.8,156.0,...,2457.0,16.0,4157.0,27.0,3075.0,20.0,1919.0,12.5,1400.0,9.1
4,4469,241,6918,182,266,3528,11135,83.0,1.9,104.0,...,888.0,20.6,871.0,20.2,592.0,13.7,308.0,7.1,315.0,7.3


**Osservazioni iniziali:**
- Il dataset contiene **28.955 righe** e **27 colonne**.
- Le prime 7 colonne (`mal_id` + conteggi utenti) sono complete senza null.
- Le 20 colonne `score_N_*` presentano 430 null, corrispondenti ad anime senza voti registrati su MAL (null strutturali).
- I tipi di dati saranno verificati nell'analisi per colonna.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [230]:
n_originale = len(df_stats)

mask_dup = df_stats.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_stats[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_stats.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_stats):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 28,955
Righe dopo la rimozione     : 28,955


Nessun duplicato esatto trovato. Tutte le 28.955 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `mal_id`

Questa colonna è la **chiave primaria** del dataset (ogni riga corrisponde a un anime distinto) ed è anche una **chiave esterna** che referenzia `mal_id` di `details.csv`.

I controlli rilevanti sono:
- **Valori nulli**: non ammessi su una chiave primaria.
- **Duplicati**: non ammessi su una chiave primaria.
- **Integrità referenziale**: ogni ID deve esistere in `details_clean.csv`.

Usiamo `check_fk` per verificare i controlli di integrità referenziale.

In [231]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_mal = check_fk(df_stats['mal_id'], df_details['mal_id'], child_df=df_stats)

print(f'Null in mal_id      : {df_stats["mal_id"].isna().sum()}')
print(f'Duplicati in mal_id : {df_stats["mal_id"].duplicated().sum()}')


  Colonna FK  (tabella figlia)        mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       28,955
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            28,955  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              28,955  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in mal_id      : 0
Duplicati in mal_id : 0


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Nessun duplicato**: la colonna è già una chiave primaria univoca.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia necessaria. 

### 2.2 `watching`

Numero di utenti che stanno attualmente guardando l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [232]:
analyze(df_stats['watching'])


  Nome serie                     watching
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           321  (1.11%)
  Positivi                       28,634   (98.89%)
  Negativi                       0   (0.00%)
  Valori unici                   5,077  (17.53%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            1,838,015
  Range                          1,838,015
  Media                          2,586.64
  Mediana                        57.0000
  Moda/e                         4
  Dev. standard      

**Osservazioni:**
- Nessun valore null. Il dtype è già `int64` che è adeguato.
- I valori sono ≥ 0 come atteso per un conteggio. Stampiamo i valori estremi per verificare se si tratta di un'anomalia. 

In [233]:
df_details_url = pd.read_csv('../datasets/details.csv', usecols=['mal_id', 'title', 'url'])
estremi_watching = (
    df_stats[df_stats["watching"] >= 275702][["mal_id", "watching", "completed", "total"]]
    .sort_values("watching", ascending=False)
    .merge(df_details_url, on="mal_id", how="left")
)
print(f"Righe con watching >= 275,702: {len(estremi_watching)}")
estremi_watching

Righe con watching >= 275,702: 9


,mal_id,watching,completed,total,title,url
0,21,1838015,14,2581610,One Piece,https://myanimelist.net/anime/21/One_Piece
1,34572,507711,737466,1832019,Black Clover,https://myanimelist.net/anime/34572/Black_Clover
2,1735,449814,1775074,2668197,Naruto: Shippuuden,https://myanimelist.net/anime/1735/Naruto__Shi...
3,34566,376388,87794,913313,Boruto: Naruto Next Generations,https://myanimelist.net/anime/34566/Boruto__Na...
4,11061,375910,2109985,3081045,Hunter x Hunter (2011),https://myanimelist.net/anime/11061/Hunter_x_H...
5,269,366847,1102401,2135419,Bleach,https://myanimelist.net/anime/269/Bleach
6,40748,358442,2234604,2880368,Jujutsu Kaisen,https://myanimelist.net/anime/40748/Jujutsu_Ka...
7,40028,293191,1697177,2202042,Shingeki no Kyojin: The Final Season,https://myanimelist.net/anime/40028/Shingeki_n...
8,5114,278645,2586001,3576614,Fullmetal Alchemist: Brotherhood,https://myanimelist.net/anime/5114/Fullmetal_A...


I valori estremi corrispondono ad anime molto famosi e qundi non si tratta di anomalie. 

Nessuna pulizia necessaria. 

### 2.3 `completed`

Numero di utenti che hanno completato l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [234]:
analyze(df_stats['completed'])


  Nome serie                     completed
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           702  (2.42%)
  Positivi                       28,253   (97.58%)
  Negativi                       0   (0.00%)
  Valori unici                   10,080  (34.81%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            3,716,436
  Range                          3,716,436
  Media                          24,838.18
  Mediana                        431.0000
  Moda/e                         0
  Dev. standard  

**Osservazioni:**
- Nessun null. Il dtype già `int64` che è adeguato.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**

### 2.4 `on_hold`

Numero di utenti che hanno messo l'anime in pausa.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [235]:
analyze(df_stats['on_hold'])


  Nome serie                     on_hold
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           1,487  (5.14%)
  Positivi                       27,468   (94.86%)
  Negativi                       0   (0.00%)
  Valori unici                   3,764  (13.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            308,583
  Range                          308,583
  Media                          974.9353
  Mediana                        22.0000
  Moda/e                         1
  Dev. standard         

**Osservazioni:**
- Nessun null. Dtype già `int64`.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**

### 2.5 `dropped`

Numero di utenti che hanno abbandonato l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [236]:
analyze(df_stats['dropped'])


  Nome serie                     dropped
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           450  (1.55%)
  Positivi                       28,505   (98.45%)
  Negativi                       0   (0.00%)
  Valori unici                   4,070  (14.06%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            237,819
  Range                          237,819
  Media                          1,301.31
  Mediana                        85.0000
  Moda/e                         0, 28
  Dev. standard       

**Osservazioni:**
- Nessun null. Dtype già `int64`.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**

### 2.6 `plan_to_watch`

Numero di utenti che intendono guardare l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [237]:
analyze(df_stats['plan_to_watch'])


  Nome serie                     plan_to_watch
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       28,955   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   9,120  (31.50%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────



  Min                            2
  Max                            687,110
  Range                          687,108
  Media                          9,058.57
  Mediana                        335.0000
  Moda/e                         18
  Dev. standard                  30,335.00
  Varianza                       920,212,256
  Errore std (SEM)               178.2716
  IQR  (Q3 − Q1)                 3,005.50
  Q1  (25° pct)                  52.0000
  Q3  (75° pct)                  3,057.50
  P5  ( 5° pct)                  17.0000
  P95 (95° pct)                  49,714.50
  Coefficiente di variazione     334.88%

Analisi outlier
────────────────────────────────────────────────────────────────────────────────

  Soglia bassa  (Q1 − 1.5×IQR)   -4,456.25
  Soglia alta   (Q3 + 1.5×IQR)   7,565.75
  Outlier rilevati               4,937  (17.05%)

  Top 5 outlier estremi (per valore assoluto):
    [23288]  687,110
    [6593]  529,859
    [4253]  493,291
    [19488]  484,134
    [16162]  483,053

**Osservazioni:**
- Nessun null. Dtype già `int64`.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**

### 2.7 `total`

Totale utenti che hanno l'anime in lista. Deve corrispondere alla somma delle cinque categorie precedenti.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [238]:
analyze(df_stats['total'])


  Nome serie                     total
  dtype                          int64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,955  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       28,955   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   12,265  (42.36%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            23
  Max                            4,230,824
  Range                          4,230,801
  Media                          38,759.63
  Mediana                        1,077.00
  Moda/e                         80
  Dev. standard     

**Osservazioni:**
- Nessun null. Dtype già `int64`.
- I valori sono ≥ 0 come atteso per un conteggio.
- Verifichiamo che `total` sia uguale alla somma delle cinque categorie utente.

In [239]:
count_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch', 'total']
somma_categorie = df_stats[['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']].sum(axis=1)
discrepanze = (somma_categorie != df_stats['total']).sum()
print(f'Righe dove total ≠ somma categorie: {discrepanze:,}')

Righe dove total ≠ somma categorie: 0


Non ci sono righe dove `total` non corrisponde alla somma delle categorie. 

Nessuna pulizia necessaria. 

### 2.8 Colonne `score_N_votes` e `score_N_percentage` (N = 1–10)

Tutte e 20 le colonne presentano lo stesso numero di null (430). Verifichiamo se si tratta delle stesse righe, ovvero anime che non hanno ancora ricevuto voti su MAL. 

In [240]:
score_votes_cols = [
    "score_1_votes", "score_2_votes", "score_3_votes", "score_4_votes", "score_5_votes",
    "score_6_votes", "score_7_votes", "score_8_votes", "score_9_votes", "score_10_votes"
]
score_pct_cols = [
    "score_1_percentage", "score_2_percentage", "score_3_percentage", "score_4_percentage", "score_5_percentage",
    "score_6_percentage", "score_7_percentage", "score_8_percentage", "score_9_percentage", "score_10_percentage"
]
score_cols = score_votes_cols + score_pct_cols

# Verifica che i null siano nelle stesse righe per tutti i campi score
mask_no_score = df_stats["score_1_votes"].isna()
tutti_stessi = df_stats[score_cols].isna().eq(mask_no_score, axis=0).all(axis=None)
print(f"Righe senza dati di voto                          : {mask_no_score.sum():,}")
print(f"Null nelle stesse righe per tutti i campi score   : {tutti_stessi}")
print()
print("Campione di 10 righe con score nulli:")
df_stats[mask_no_score][["mal_id", "watching", "completed", "on_hold", "dropped", "plan_to_watch", "total"]].sample(10, random_state=42)

Righe senza dati di voto                          : 430
Null nelle stesse righe per tutti i campi score   : True

Caratteristiche anime senza score (colonne di conteggio):

Campione di 10 righe con score nulli:


,mal_id,watching,completed,on_hold,dropped,plan_to_watch,total
28620,35854,0,0,0,0,1920,1920
5005,61269,4,0,0,0,10128,10132
11463,62050,1,0,0,0,1322,1323
1983,62430,0,0,0,0,1721,1721
24779,61014,0,0,0,0,2966,2966
26039,61073,0,0,0,0,333,333
10296,56906,1,1,1,0,29085,29088
9920,60831,0,0,0,0,1488,1488
10828,61567,0,0,0,0,101,101
11253,62507,0,0,0,0,394,394


Le 430 righe con valutazione nullo sono le stesse in tutte le colonne. Questo è strutturale e si riferisce ad anime che non hanno ancora ricevuto nessun voto. Vanno mantenute perchè le righe forniscono dati sulle altre colonne non nulle.  

Usiamo `analyze` per continuare la verifica sul resto delle righe. I duplicati sono **attesi**: diversi anime possono ricevere lo stesso numero di voti o la stessa percentuale. 

In [241]:
_ = df_stats[score_cols].apply(analyze)


  Nome serie                     score_1_votes
  dtype                          float64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,525  (98.51%)
  Null / NaN                     430  (1.49%)
  Zeri                           1,214  (4.19%)
  Positivi                       27,311   (94.32%)
  Negativi                       0   (0.00%)
  Valori unici                   1,486  (5.13%)
  Valori float                   0

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0.0000
  Max                            50,279.00
  Range                          50,279.00
  Media                          143.4331
  Mediana                        17.0000
  Moda/

              1,317.00            1     0.0%
              2,979.00            1     0.0%
              745.0000            1     0.0%
              4,021.00            1     0.0%
              353.0000            1     0.0%


  Nome serie                     score_2_votes
  dtype                          float64
  Dimensione                     28,955
  Range indice                   0 … 28954
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   28,955
  Valori non nulli               28,525  (98.51%)
  Null / NaN                     430  (1.49%)
  Zeri                           5,240  (18.10%)
  Positivi                       23,285   (80.42%)
  Negativi                       0   (0.00%)
  Valori unici                   1,464  (5.06%)
  Valori float                   0

Statistiche descrittive
───────────────────────────────────────────────────────────────────────

**Pulizia necessaria:**  
- Le colonne `score_N_votes` hanno dtype `float64` a causa dei 430 null. Poiché rappresentano conteggi interi, li convertiamo in `Int64`.
- Le colonne `score_N_percentage` rimangono `float64` in quanto solo valori percentuali.

In [242]:
# Conversione score_N_votes da float64 a Int64 (nullable integer)
df_stats[score_votes_cols] = df_stats[score_votes_cols].astype('Int64')

print('Dtype score_N_votes dopo la conversione:')
print(df_stats[score_votes_cols].dtypes)

Dtype score_N_votes dopo la conversione:
score_1_votes     Int64
score_2_votes     Int64
score_3_votes     Int64
score_4_votes     Int64
score_5_votes     Int64
score_6_votes     Int64
score_7_votes     Int64
score_8_votes     Int64
score_9_votes     Int64
score_10_votes    Int64
dtype: object


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [243]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_stats):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_stats):>10,}')
print()
df_stats.to_csv('../datasets_cleaned/stats_clean.csv', index=False)
print('Salvato: datasets_cleaned/stats_clean.csv')

=== Riepilogo Dataset Pulito ===
Righe originali      :     28,955
Righe dopo cleaning  :     28,955
Righe rimosse totali :          0



Salvato: datasets_cleaned/stats_clean.csv
